In [8]:
# Cell 1: config, then a one-time conversion of the CSVs to Parquet (needs: pip install pyarrow)
import numpy as np
import pandas as pd
from pathlib import Path

# ---- config: edit these -----------------------------------------------------
PQ_DIR   = Path("../data/TEP")     # where the Parquet copies are written
FILES = {                      # adjust the file names if yours differ
    ("train", "free"):   "FaultFree_Training.csv",
    ("train", "faulty"): "Faulty_Training.csv",
    ("test",  "free"):   "FaultFree_Testing.csv",
    ("test",  "faulty"): "Faulty_Testing.csv",
}
FAULTS  = [1, 4, 8, 13, 14]    # fault classes to keep (class 0 = normal is always included)
N_TRAIN = 50                   # runs per class used for training
N_VAL   = 10                   # further runs per class, held out for validation
N_TEST  = 50                   # runs per class taken from the test files
CHUNK   = 500_000              # rows per read while converting; lower this if memory is tight
# ------------------------------------------------------------------------------

META = ["faultNumber", "simulationRun", "sample"]
FEATURES = [f"xmeas_{i}" for i in range(1, 42)] + [f"xmv_{i}" for i in range(1, 12)]  # 52 variables
DTYPES = {**{c: "float32" for c in FEATURES}, **{c: "int16" for c in META}}

In [9]:
# Cell 2: read only the chosen subset from Parquet, reshape to (runs, time, 52), cache it
def read_filtered(kind, which, faults, max_run):
    """Parquet only reads the columns and row groups that can match the filter."""
    path = PQ_DIR / FILES[(kind, which)].replace(".csv", ".parquet")
    df = pd.read_parquet(path, columns=META + FEATURES,
                         filters=[("faultNumber", "in", faults), ("simulationRun", "<=", max_run)])
    return df.sort_values(META, ignore_index=True)


def to_runs(df):
    """Long table -> array of shape (runs, time, 52), plus the fault number and run id of each run."""
    sizes = df.groupby(["faultNumber", "simulationRun"], sort=True).size()
    T = int(sizes.iloc[0])
    assert (sizes == T).all(), "runs have different lengths"
    sample = df["sample"].to_numpy().reshape(-1, T)
    assert (sample == np.arange(1, T + 1)).all(), "sample is not 1..T inside every run"
    X = df[FEATURES].to_numpy(np.float32).reshape(-1, T, len(FEATURES))
    keys = sizes.index.to_frame(index=False)
    return X, keys["faultNumber"].to_numpy(np.int16), keys["simulationRun"].to_numpy(np.int16)


def load_kind(kind, max_run):
    free = to_runs(read_filtered(kind, "free", [0], max_run))
    faulty = to_runs(read_filtered(kind, "faulty", FAULTS, max_run))
    return tuple(np.concatenate(pair) for pair in zip(free, faulty))


def select(X, fault, run, lo, hi):
    m = (run > lo) & (run <= hi)
    return {"X": X[m], "fault": fault[m], "run": run[m]}


tag = f"f{'-'.join(map(str, FAULTS))}_tr{N_TRAIN}_va{N_VAL}_te{N_TEST}"
cache = Path("../cache") / f"tep_{tag}.npz"
cache.parent.mkdir(exist_ok=True)

if cache.exists():                                   # fastest path: the finished arrays
    z = np.load(cache)
    data = {s: {k: z[f"{s}_{k}"] for k in ("X", "fault", "run")} for s in ("train", "val", "test")}
else:                                                # otherwise build them from Parquet
    Xa, fa, ra = load_kind("train", N_TRAIN + N_VAL)
    Xb, fb, rb = load_kind("test", N_TEST)
    data = {
        "train": select(Xa, fa, ra, 0, N_TRAIN),
        "val":   select(Xa, fa, ra, N_TRAIN, N_TRAIN + N_VAL),   # held-out runs, same files
        "test":  select(Xb, fb, rb, 0, N_TEST),
    }
    np.savez(cache, **{f"{s}_{k}": v for s, d in data.items() for k, v in d.items()})

for s, d in data.items():
    classes, counts = np.unique(d["fault"], return_counts=True)
    print(f"{s:5s} X{d['X'].shape}  {d['X'].nbytes / 1e6:.0f} MB  runs per class: "
          f"{dict(zip(classes.tolist(), counts.tolist()))}")


train X(300, 500, 52)  31 MB  runs per class: {0: 50, 1: 50, 4: 50, 8: 50, 13: 50, 14: 50}
val   X(60, 500, 52)  6 MB  runs per class: {0: 10, 1: 10, 4: 10, 8: 10, 13: 10, 14: 10}
test  X(300, 960, 52)  60 MB  runs per class: {0: 50, 1: 50, 4: 50, 8: 50, 13: 50, 14: 50}


In [10]:
# Cell 2: standardise, then index windows without storing copies of them
W, STRIDE = 30, 10                                   # window length and stride, in rows (3 min each)
ONSET = {"train": 20, "val": 20, "test": 160}        # fault-free rows at the start of faulty runs; verify on your copy
CLASSES = [0] + sorted(FAULTS)                       # labels below are indices into this list

# per-variable mean/std from fault-free TRAINING runs only (no leakage from val/test)
train = data["train"]
normal = train["X"][train["fault"] == 0].reshape(-1, train["X"].shape[-1])
mu = normal.mean(0).astype(np.float32)
sd = normal.std(0).astype(np.float32)
sd[sd == 0] = 1.0

class Windows:
    """Index of (run, end-row) pairs; each window is sliced out on demand."""

    def __init__(self, split, w=W, stride=STRIDE):
        d, onset = data[split], ONSET[split]
        self.w = w
        self.X = (d["X"] - mu) / sd                  # float32, shape (runs, time, 52)
        R, T, _ = self.X.shape
        grid = np.meshgrid(np.arange(R), np.arange(w - 1, T, stride), indexing="ij")
        run_i, end_i = (g.ravel() for g in grid)
        fault = d["fault"][run_i]
        pre = (fault == 0) | (end_i < onset)         # fault-free run, or window ends before onset
        post = (fault > 0) & (end_i - w + 1 >= onset)  # window lies entirely after onset
        keep = pre | post                            # windows straddling onset are dropped
        self.run_i, self.end_i = run_i[keep], end_i[keep]
        self.y = np.searchsorted(CLASSES, np.where(post, fault, 0)[keep]).astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        r, e = self.run_i[i], self.end_i[i]
        return self.X[r, e - self.w + 1: e + 1].T, self.y[i]   # (52 nodes, w), class index

    def batch(self, idx):
        return np.stack([self[i][0] for i in idx]), self.y[idx]  # (B, 52, w), (B,)

sets = {s : Windows(s) for s in ("train", "val", "test")}
for s, ds in sets.items():
    print(f"{s:5s} {len(ds):6d} windows; per class {dict(zip(CLASSES, np.bincount(ds.y, minlength=len(CLASSES)).tolist()))}")


train  13900 windows; per class {0: 2400, 1: 2300, 4: 2300, 8: 2300, 13: 2300, 14: 2300}
val     2780 windows; per class {0: 480, 1: 460, 4: 460, 8: 460, 13: 460, 14: 460}
test   27700 windows; per class {0: 8200, 1: 3900, 4: 3900, 8: 3900, 13: 3900, 14: 3900}


In [17]:
print(sets["train"].batch([0, 1, 2]))

(array([[[-0.00404039,  0.01881199, -0.00404039, ..., -0.08386545,
         -0.9268515 , -0.8531422 ],
        [ 0.31613198, -0.11058626, -0.0842776 , ..., -0.04628097,
          1.2601681 ,  0.33658952],
        [ 0.4981937 ,  1.1968763 , -0.7979134 , ...,  0.26023993,
         -0.7548738 , -0.2485839 ],
        ...,
        [-0.17981242, -0.20965087, -0.1949113 , ...,  0.06177101,
          0.17860834,  0.25661922],
        [ 0.2888002 ,  0.4745091 ,  0.18032216, ..., -0.55331105,
          0.3090285 , -0.22786283],
        [ 0.22450249, -0.63291746,  1.649886  , ..., -0.01157961,
          0.40515596, -0.4084698 ]],

       [[-0.5055194 , -0.4765508 ,  0.19262363, ...,  1.7298907 ,
          0.10443048,  0.11923662],
        [ 0.4155012 , -0.57236844,  0.49442002, ...,  0.7223855 ,
         -1.2884319 , -1.8145194 ],
        [-0.49413943,  0.2045431 , -0.07645027, ..., -1.0282654 ,
          0.882979  , -0.5726045 ],
        ...,
        [-0.24524088, -0.18268819, -0.21612103, ..., 